# AI Price Prediction Model — Machine Learning Pipeline

**Goal**: Predict recyclable material price per kg from physical attributes, demand signals, and logistics factors.

**Target**: `price_per_kg` (INR)


## 1. Environment Setup & Data Loading


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib


In [ ]:
df = pd.read_csv('../datasets/material_prices.csv')
print(f'Loaded {len(df)} samples')
df.head()


## 2. Feature Preprocessing Pipeline


In [ ]:
NUM_FEATURES = ['weight_kg', 'historical_price', 'processing_cost', 'transportation_distance', 'month', 'buyer_demand']
CAT_FEATURES = ['material_type', 'quality', 'location', 'demand_level', 'seller_type', 'material_condition']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), NUM_FEATURES),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_FEATURES)
])


## 3. Train / Validation / Test Splitting (Holdout)


In [ ]:
X = df[NUM_FEATURES + CAT_FEATURES]
y = df['price_per_kg']

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=42)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')


## 4. Model Training & Comparison


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, random_state=42)
}

results = {}
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)
    results[name] = {'Val MAE': mae, 'Val R2': r2}
    print(f'{name} -> Val MAE: {mae:.2f}, Val R²: {r2:.4f}')


## 5. Model Evaluation on Test Set & Artifact Export


In [ ]:
best_pipe = Pipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, random_state=42))])
best_pipe.fit(X_train_val, y_train_val)
test_preds = best_pipe.predict(X_test)
print(f'Final Test MAE: {mean_absolute_error(y_test, test_preds):.2f}')
print(f'Final Test R²:  {r2_score(y_test, test_preds):.4f}')

os.makedirs('../ml/models', exist_ok=True)
joblib.dump(best_pipe, '../ml/models/price_model_v1.pkl')
print('Saved price_model_v1.pkl successfully')
